# 01 · Extracción, limpieza y normalización
### Riesgo de acceso a la salud por municipio en Colombia · Fase 1

Este notebook **narra** paso a paso lo que hacen `make extract` y `make clean`. No
copia la lógica: importa las mismas funciones del paquete `src/salud/`, así que lo
que se ve aquí es exactamente lo que corre en la terminal y en Docker.

| Paso | Módulo | Comando |
|---|---|---|
| Extracción desde datos.gov.co | `salud.data.extract` | `make extract` |
| Limpieza y validaciones | `salud.data.clean` | `make clean` |
| Tabla municipal | `salud.features` | `make features` |

Los parámetros (fecha de corte, EPS intervenidas, umbrales) están en
`src/salud/config.py`. Para cambiarlos se edita ese archivo, no el notebook.

In [ ]:
# Solo si se ejecuta en Google Colab: descomente estas líneas.
# Clonan el repo, instalan el paquete y guardan los datos en Drive para no perderlos.
# from google.colab import drive; drive.mount("/content/drive")
# !git clone -q https://github.com/DABO-lab/riesgo-acceso-salud.git
# !pip install -q -e riesgo-acceso-salud
# import os; os.environ["SALUD_DATA_DIR"] = "/content/drive/MyDrive/proyecto2_salud/data"

In [ ]:
import pandas as pd

from salud import config
from salud.data import clean, extract

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Datos en:", config.DATA_DIR)
print("EPS intervenidas:", sorted(config.EPS_INTERVENIDAS))

## 1. Extracción desde la API de datos abiertos

`extract.main()` descarga 7 tablas y las guarda en `data/raw/<fecha>/`. Si el
archivo del día ya existe, lo lee de disco: el análisis no depende de que el portal
esté disponible y cada corte queda guardado para compararlo después.

- La BDUA se pide **agregada en el servidor** en dos vistas: por EPS
  (municipio × EPS × régimen × zona × estado) y demográfica (municipio × régimen ×
  sexo × grupo de edad).
- Del REPS **no** se piden correos, teléfonos ni documentos (Ley 1581 de 2012).

In [ ]:
extract.main()
fecha = config.ultimo_corte()
raw = clean.cargar_raw(fecha)
{nombre: len(df) for nombre, df in raw.items()}

> **Resultado del corte 2026-10-25.** Se descargaron unas 127.000 filas en lugar de
> casi 2 millones: BDUA por EPS 28.301 (contributivo) y 25.333 (subsidiado), BDUA
> demográfica 26.102 y 26.942, REPS 19.551 sedes de IPS y DIVIPOLA 1.122 municipios.

## 2. Perfilamiento de calidad
Antes de transformar nada se revisa cada tabla: tipos, nulos, duplicados y cardinalidad.

In [ ]:
def perfil(df, nombre):
    p = pd.DataFrame(
        {
            "tipo": df.dtypes.astype(str),
            "nulos": df.isna().sum(),
            "pct_nulos": (df.isna().mean() * 100).round(2),
            "valores_unicos": df.nunique(),
            "ejemplo": df.iloc[0] if len(df) else None,
        }
    )
    print(
        f"\n=== {nombre}: {len(df):,} filas · {df.shape[1]} columnas · duplicados: {df.duplicated().sum():,}"
    )
    display(p)


for nombre, df in raw.items():
    perfil(df, nombre)

In [ ]:
# Valores de las categóricas clave: aquí aparecen las inconsistencias a corregir
eps_crudo = pd.concat([raw["bdua_eps_contributivo"], raw["bdua_eps_subsidiado"]])
for col in ["tps_rgm_nombre", "zns_nombre", "tps_est_afl_nombre"]:
    print(col, "→", sorted(eps_crudo[col].dropna().unique()))

# Control de totales: deben cuadrar con lo publicado (~23 M contributivo, ~27 M subsidiado)
for k in ["bdua_eps_contributivo", "bdua_eps_subsidiado", "bdua_demo_contributivo", "bdua_demo_subsidiado"]:
    print(f"{k:<24} total afiliados: {pd.to_numeric(raw[k]['afiliados']).sum():>14,.0f}")

> **Lo que se encontró en el corte 2026-10-25**
>
> - Ninguna tabla tiene nulos ni filas duplicadas.
> - `afiliados` llega como texto: se convierte a número en la limpieza.
> - DIVIPOLA tiene 1.122 municipios pero solo 1.037 nombres distintos (varios
>   "San Pedro" o "Bolívar"): la unión debe ser por **departamento + municipio**.
> - Las coordenadas de DIVIPOLA usan coma decimal (`6,246631`).
> - La zona tiene 6 nombres distintos y el estado del afiliado dos ("Activo" y
>   "Protección Laboral", ambos con cobertura, así que se cuentan los dos).

## 3. Limpieza y normalización
### 3.1 Normalización de texto
Mayúsculas, sin tildes ni signos, y corrección del error `¥` por `Ñ` de la BDUA.
Las pruebas de esta función están en `tests/test_clean.py`.

In [ ]:
for ejemplo in ["NARI¥O", "Bogotá, D.C.", "  la paz  robles "]:
    print(f"{ejemplo!r:<22} → {clean.normalizar_texto(ejemplo)!r}")

### 3.2 Homologación de municipios con el código DIVIPOLA
El REPS ya trae el código DIVIPOLA, pero la BDUA solo trae nombres. El cruce
(`clean.asignar_divipola`) se hace en este orden:

1. **Tabla de homologación manual** (`src/salud/data/catalogos.py`), revisada uno a
   uno: `CUCUTA` → `SAN JOSE DE CUCUTA`, `BUGA` → `GUADALAJARA DE BUGA`,
   `SAN PEDRO` (Antioquia) → `SAN PEDRO DE LOS MILAGROS`...
2. **Cruce exacto** por departamento + municipio, con alias de departamento
   (`VALLE` → `VALLE DEL CAUCA`).
3. `DPTO. GUAINIA` y `DPTO. VAUPES` son **áreas no municipalizadas**: se reportan
   y quedan fuera de la tabla municipal.

In [ ]:
div = clean.preparar_divipola(raw["divipola"])
bdua_eps = clean.limpiar_bdua_eps(raw["bdua_eps_contributivo"], raw["bdua_eps_subsidiado"], div)
display(clean.reporte_cruce(bdua_eps))

pendientes = bdua_eps.loc[bdua_eps["estado_cruce"] == "sin_cruce", ["dpr_nombre", "mnc_nombre"]]
if len(pendientes):
    print("Municipios sin cruce. Agréguelos a HOMOLOGACION_MANUAL en catalogos.py:")
    display(pendientes.drop_duplicates())
else:
    print("Todos los municipios de la BDUA quedaron homologados con DIVIPOLA.")

> **Resultado del corte 2026-10-25.** 987 municipios cruzaron por nombre exacto
> (46,5 millones de afiliados, 92,8 %) y 54 por homologación manual (3,6 millones,
> 7,2 %). Sin la homologación se habrían perdido Cúcuta, Cartagena, Tumaco y Buga.

### 3.3 Unificación de EPS y marcación de intervenidas
Una EPS aparece con varios códigos: uno por régimen y otro con `-CM` (movilidad).
`clean.eps_canonica` les asigna un nombre corto con las reglas de `catalogos.py`.

In [ ]:
resumen_eps = (
    bdua_eps.groupby(["eps", "eps_intervenida"], as_index=False)
    .agg(afiliados=("afiliados", "sum"), codigos=("ent_id", lambda s: ", ".join(sorted(s.unique()))))
    .sort_values("afiliados", ascending=False)
)
resumen_eps["pct_nacional"] = resumen_eps["afiliados"] / resumen_eps["afiliados"].sum() * 100
resumen_eps

> **Resultado del corte 2026-10-25.** Unos 60 códigos quedaron en 28 EPS. Nueva EPS
> es la más grande (11,3 millones, 22,5 %) y cuatro EPS (Nueva EPS, Sanitas, Sura y
> Salud Total) concentran el 57 % de los afiliados.

### 3.4 Todo junto, con validaciones
`clean.limpiar_corte` aplica todos los pasos (incluida la limpieza del REPS: solo
sedes de IPS, sin duplicados y con un código de municipio que exista) y luego
**valida**. Si algo no cuadra, se detiene con un mensaje que dice qué revisar:

- afiliados no nulos ni negativos y solo dos regímenes;
- ningún grupo de edad desconocido;
- todas las EPS intervenidas aparecen en la BDUA;
- ningún municipio sin cruce;
- las dos vistas de la BDUA suman exactamente lo mismo.

Es lo mismo que ejecutar `make clean`.

In [ ]:
limpios = clean.limpiar_corte(raw)

> **Resultado del corte 2026-10-25.** Validaciones superadas con 50.137.787
> afiliados: la limpieza no perdió ni duplicó personas. El REPS quedó con 19.551
> sedes de IPS (se excluyeron 4 con el código 27086, que no existe en DIVIPOLA).

## 4. Guardar y construir la tabla municipal
Los mismos dos comandos de la terminal. El resultado queda en `data/processed/`.

In [ ]:
from salud import features

clean.main()
features.main()

Siguiente notebook: **`02-eda.ipynb`**, que parte de la tabla municipal.